# 05 — Modelo de Redes Neurais com MLlib

Este notebook treina modelos de **Redes Neurais** usando **PySpark MLlib** para classificar a faixa térmica futura no estado de São Paulo.

Diferente dos notebooks de Regressão Linear e Random Forest, que preveem temperatura em graus Celsius, aqui o problema será tratado como **classificação**.

Serão treinados dois modelos:

1. **Classificação da faixa térmica de amanhã**.
2. **Classificação da faixa térmica média dos próximos 7 dias**.

Este notebook mantém o mesmo padrão dos notebooks anteriores:

- uso de **MLlib** para o modelo;
- uso de **Spark SQL** para consultas, criação de alvos, joins, agregações e análises;
- uso de **Plotly** para gráficos;
- sem uso de `toPandas()`.

O modelo utilizado será o **Multilayer Perceptron Classifier**, uma rede neural disponível no MLlib para tarefas de classificação.

## 1. Por que transformar temperatura em categorias?

O PySpark MLlib possui uma rede neural nativa chamada `MultilayerPerceptronClassifier`.

Esse modelo é voltado para **classificação**, não para regressão contínua. Como a variável temperatura é originalmente numérica, será criada uma versão categórica do alvo.

Assim, em vez de responder:

> qual será a temperatura exata em °C?

a rede neural responderá:

> qual será a faixa térmica esperada?

As faixas usadas neste notebook serão definidas com base nos quartis da base de treino.

Essa escolha evita criar classes muito desbalanceadas e reduz o risco de o modelo aprender apenas a classe majoritária.

Os cortes são calculados separadamente para cada objetivo:

- `temperatura_amanha`;
- `temperatura_media_proximos_7_dias`.

As classes serão:

| Classe numérica | Categoria | Interpretação |
|---:|---|---|
| 0 | mais_fria | valores abaixo do 1º quartil |
| 1 | amena | valores entre o 1º e o 2º quartil |
| 2 | quente | valores entre o 2º e o 3º quartil |
| 3 | mais_quente | valores acima do 3º quartil |

Os quartis são calculados apenas na base de treino e depois aplicados também na base de teste, evitando vazamento de dados.

## 2. Imports e inicialização da SparkSession

In [26]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

import plotly.express as px
import plotly.graph_objects as go

In [27]:
spark = (
    SparkSession.builder
    .appName("05_modelo_redes_neurais_mllib")
    .getOrCreate()
)

spark

## 3. Caminhos das bases Parquet

As bases finais já foram criadas no notebook de pré-processamento.

Também serão carregadas as bases completas, porque elas ainda possuem colunas interpretáveis como `tipo_area`, `macro_regiao_sp` e `faixa_altitude`.

In [28]:
amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
amanha_test_path  = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
semana_test_path  = "/home/jovyan/work/data/processed/weather_sp_semana_test"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"

modelos_path = "/home/jovyan/work/models"
resultados_path = "/home/jovyan/work/data/processed"

## 4. Carregamento das bases

As bases são lidas diretamente do formato Parquet e registradas como views temporárias para uso com `spark.sql`.

In [29]:
amanha_train = spark.read.parquet(amanha_train_path)
amanha_test = spark.read.parquet(amanha_test_path)

semana_train = spark.read.parquet(semana_train_path)
semana_test = spark.read.parquet(semana_test_path)

dataset_amanha_completo = spark.read.parquet(dataset_amanha_path)
dataset_semana_completo = spark.read.parquet(dataset_semana_path)

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")
semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")
dataset_amanha_completo.createOrReplaceTempView("dataset_amanha_completo")
dataset_semana_completo.createOrReplaceTempView("dataset_semana_completo")

In [30]:
spark.sql("""
    SELECT 'amanha_train' AS base, COUNT(*) AS total_linhas FROM amanha_train
    UNION ALL
    SELECT 'amanha_test' AS base, COUNT(*) AS total_linhas FROM amanha_test
    UNION ALL
    SELECT 'semana_train' AS base, COUNT(*) AS total_linhas FROM semana_train
    UNION ALL
    SELECT 'semana_test' AS base, COUNT(*) AS total_linhas FROM semana_test
""").show(truncate=False)

+------------+------------+
|base        |total_linhas|
+------------+------------+
|amanha_train|120232      |
|amanha_test |45940       |
|semana_train|120232      |
|semana_test |45940       |
+------------+------------+



## 5. Cache estratégico dos datasets de modelagem

No pré-processamento, o cache foi evitado para reduzir consumo de memória.

Aqui, no notebook de modelagem, o cache faz sentido porque os mesmos datasets serão usados várias vezes.

In [31]:
amanha_train = amanha_train.cache()
amanha_test = amanha_test.cache()
semana_train = semana_train.cache()
semana_test = semana_test.cache()

amanha_train.count()
amanha_test.count()
semana_train.count()
semana_test.count()

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")
semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")

## 6. Conferência dos schemas

In [32]:
amanha_train.printSchema()
semana_train.printSchema()

root
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- data_formatada: date (nullable = true)
 |-- ano_imputado: integer (nullable = true)
 |-- mes_sin_imputado: double (nullable = true)
 |-- mes_cos_imputado: double (nullable = true)
 |-- latitude_imputado: double (nullable = true)
 |-- longitude_imputado: double (nullable = true)
 |-- altitude_imputado: double (nullable = true)
 |-- temp_media_dia_imputado: double (nullable = true)
 |-- temp_min_dia_imputado: double (nullable = true)
 |-- temp_max_dia_imputado: double (nullable = true)
 |-- temp_orvalho_media_dia_imputado: double (nullable = true)
 |-- umidade_media_dia_imputado: double (nullable = true)
 |-- umidade_min_dia_imputado: double (nullable = true)
 |-- umidade_max_dia_imputado: double (nullable = true)
 |-- pressao_media_dia_imputado: double (nullable = true)
 |-- precipitacao_total_dia_imputado: double (nullable = true)
 |-- radiacao_media_dia_imputado: double (nullable = true)
 |-- 

## 7. Criação dos alvos categóricos com Spark SQL

Nesta etapa, os alvos numéricos são transformados em classes térmicas.

In [33]:
# Cálculo dos quartis apenas na base de treino
# Isso evita vazamento de dados, porque o teste não é usado para definir os cortes.

q1_amanha, q2_amanha, q3_amanha = amanha_train.approxQuantile(
    "temperatura_amanha",
    [0.25, 0.50, 0.75],
    0.01
)

q1_semana, q2_semana, q3_semana = semana_train.approxQuantile(
    "temperatura_media_proximos_7_dias",
    [0.25, 0.50, 0.75],
    0.01
)

print("Cortes para temperatura de amanhã:")
print(f"Q1: {q1_amanha:.2f}")
print(f"Q2: {q2_amanha:.2f}")
print(f"Q3: {q3_amanha:.2f}")

print("\nCortes para temperatura média dos próximos 7 dias:")
print(f"Q1: {q1_semana:.2f}")
print(f"Q2: {q2_semana:.2f}")
print(f"Q3: {q3_semana:.2f}")

Cortes para temperatura de amanhã:
Q1: 18.99
Q2: 21.89
Q3: 24.18

Cortes para temperatura média dos próximos 7 dias:
Q1: 19.13
Q2: 21.87
Q3: 24.03


In [34]:
amanha_train_cls = spark.sql("""
    SELECT *,
        CASE
            WHEN temperatura_amanha < 18 THEN CAST(0 AS DOUBLE)
            WHEN temperatura_amanha < 25 THEN CAST(1 AS DOUBLE)
            ELSE CAST(2 AS DOUBLE)
        END AS classe_temperatura_amanha
    FROM amanha_train
""")

amanha_test_cls = spark.sql("""
    SELECT *,
        CASE
            WHEN temperatura_amanha < 18 THEN CAST(0 AS DOUBLE)
            WHEN temperatura_amanha < 25 THEN CAST(1 AS DOUBLE)
            ELSE CAST(2 AS DOUBLE)
        END AS classe_temperatura_amanha
    FROM amanha_test
""")

semana_train_cls = spark.sql("""
    SELECT *,
        CASE
            WHEN temperatura_media_proximos_7_dias < 18 THEN CAST(0 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < 25 THEN CAST(1 AS DOUBLE)
            ELSE CAST(2 AS DOUBLE)
        END AS classe_temperatura_semana
    FROM semana_train
""")

semana_test_cls = spark.sql("""
    SELECT *,
        CASE
            WHEN temperatura_media_proximos_7_dias < 18 THEN CAST(0 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < 25 THEN CAST(1 AS DOUBLE)
            ELSE CAST(2 AS DOUBLE)
        END AS classe_temperatura_semana
    FROM semana_test
""")

amanha_train_cls.createOrReplaceTempView("amanha_train_cls")
amanha_test_cls.createOrReplaceTempView("amanha_test_cls")
semana_train_cls.createOrReplaceTempView("semana_train_cls")
semana_test_cls.createOrReplaceTempView("semana_test_cls")

## 8. Distribuição das classes

Antes de treinar uma rede neural de classificação, é importante verificar a distribuição das classes.

In [35]:
spark.sql("""
    SELECT
        classe_temperatura_amanha,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM amanha_train_cls
    GROUP BY classe_temperatura_amanha
    ORDER BY classe_temperatura_amanha
""").show(truncate=False)

spark.sql("""
    SELECT
        classe_temperatura_semana,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM semana_train_cls
    GROUP BY classe_temperatura_semana
    ORDER BY classe_temperatura_semana
""").show(truncate=False)

+-------------------------+---------+---------------+
|classe_temperatura_amanha|categoria|total_registros|
+-------------------------+---------+---------------+
|0.0                      |fria     |21637          |
|1.0                      |amena    |77581          |
|2.0                      |quente   |21014          |
+-------------------------+---------+---------------+



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `classe_temperatura_amanha` cannot be resolved. Did you mean one of the following? [`classe_temperatura_semana`, `ano_imputado`, `data_formatada`, `faixa_altitude_idx`, `latitude_imputado`].; line 5 pos 17;
'Sort ['classe_temperatura_semana ASC NULLS FIRST], true
+- 'Aggregate [classe_temperatura_semana#31994], [classe_temperatura_semana#31994, CASE WHEN ('classe_temperatura_amanha = 0) THEN fria WHEN ('classe_temperatura_amanha = 1) THEN amena WHEN ('classe_temperatura_amanha = 2) THEN quente END AS categoria#32829, count(1) AS total_registros#32830L]
   +- SubqueryAlias semana_train_cls
      +- View (`semana_train_cls`, [station#24243,station_code#24244,data_formatada#24245,ano_imputado#24246,mes_sin_imputado#24247,mes_cos_imputado#24248,latitude_imputado#24249,longitude_imputado#24250,altitude_imputado#24251,temp_media_dia_imputado#24252,temp_min_dia_imputado#24253,temp_max_dia_imputado#

Para manter o notebook coerente com Spark, os gráficos serão criados a partir de listas obtidas com `collect()` em resultados pequenos e já agregados.

In [ ]:
def spark_df_para_dicts(df):
    return [row.asDict() for row in df.collect()]

### Gráfico: distribuição das classes no treino

In [ ]:
dist_amanha = spark.sql("""
    SELECT
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM amanha_train_cls
    GROUP BY classe_temperatura_amanha
    ORDER BY classe_temperatura_amanha
""")

fig = px.bar(
    spark_df_para_dicts(dist_amanha),
    x="categoria",
    y="total_registros",
    text="total_registros",
    title="Distribuição das classes térmicas — treino amanhã",
    labels={"categoria": "Categoria térmica", "total_registros": "Total de registros"}
)

fig.update_traces(textposition="outside")
fig.show()

In [ ]:
dist_semana = spark.sql("""
    SELECT
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM semana_train_cls
    GROUP BY classe_temperatura_semana
    ORDER BY classe_temperatura_semana
""")

fig = px.bar(
    spark_df_para_dicts(dist_semana),
    x="categoria",
    y="total_registros",
    text="total_registros",
    title="Distribuição das classes térmicas — treino próximos 7 dias",
    labels={"categoria": "Categoria térmica", "total_registros": "Total de registros"}
)

fig.update_traces(textposition="outside")
fig.show()

## 9. Definição dos alvos e das features

As features serão identificadas automaticamente, removendo apenas colunas de identificação, alvos numéricos originais e alvos categóricos.

In [ ]:
coluna_alvo_amanha_original = "temperatura_amanha"
coluna_alvo_semana_original = "temperatura_media_proximos_7_dias"

coluna_label_amanha = "classe_temperatura_amanha"
coluna_label_semana = "classe_temperatura_semana"

colunas_identificacao = ["station", "station_code", "data_formatada"]

features_amanha = [
    c for c in amanha_train_cls.columns
    if c not in colunas_identificacao + [coluna_alvo_amanha_original, coluna_label_amanha]
]

features_semana = [
    c for c in semana_train_cls.columns
    if c not in colunas_identificacao + [coluna_alvo_semana_original, coluna_label_semana]
]

print(f"Features amanhã ({len(features_amanha)}):")
print(features_amanha)

print(f"\nFeatures semana ({len(features_semana)}):")
print(features_semana)

Redes neurais são sensíveis à escala das variáveis.

Por isso, antes de treinar a rede neural, as features serão montadas com `VectorAssembler` e padronizadas com `StandardScaler`.

## 11. Treinamento da rede neural para faixa térmica de amanhã

A arquitetura usada será:

- camada de entrada: número de features;
- primeira camada oculta: 32 neurônios;
- segunda camada oculta: 16 neurônios;
- camada de saída: 4 classes.

In [ ]:
num_features_amanha = len(features_amanha)
num_classes = 3

layers_amanha = [num_features_amanha, 32, 16, num_classes]

assembler_amanha = VectorAssembler(
    inputCols=features_amanha,
    outputCol="features_brutas",
    handleInvalid="keep"
)

scaler_amanha = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withMean=False,
    withStd=True
)

mlp_amanha = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=coluna_label_amanha,
    predictionCol="prediction",
    layers=layers_amanha,
    maxIter=100,
    blockSize=128,
    seed=42
)

pipeline_amanha = Pipeline(stages=[assembler_amanha, scaler_amanha, mlp_amanha])

modelo_mlp_amanha = pipeline_amanha.fit(amanha_train_cls)

pred_amanha = modelo_mlp_amanha.transform(amanha_test_cls)
pred_amanha.createOrReplaceTempView("pred_amanha")

print("Rede neural para faixa térmica de amanhã treinada com sucesso.")

## 12. Previsões do modelo de amanhã

Após o treinamento, o modelo é aplicado à base de teste.

Nesta etapa, também são criadas colunas interpretáveis para a classe real e a classe prevista.

In [ ]:
pred_amanha = spark.sql("""
    SELECT *,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria_real,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria_prevista,
        CASE
            WHEN classe_temperatura_amanha = prediction THEN 1
            ELSE 0
        END AS acertou
    FROM pred_amanha
""")

pred_amanha.createOrReplaceTempView("pred_amanha")

spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND(temperatura_amanha, 2) AS temperatura_real,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_amanha
    LIMIT 20
""").show(truncate=False)

## 13. Função de avaliação de classificação

Serão usadas as seguintes métricas:

- **Accuracy:** percentual geral de acertos.
- **F1-score:** métrica que combina precisão e revocação.
- **Weighted Precision:** precisão ponderada pelo tamanho das classes.
- **Weighted Recall:** revocação ponderada pelo tamanho das classes.

In [ ]:
def avaliar_classificacao(predicoes, coluna_label, nome_modelo):
    avaliador_accuracy = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="accuracy"
    )

    avaliador_f1 = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="f1"
    )

    avaliador_precision = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    avaliador_recall = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    accuracy = avaliador_accuracy.evaluate(predicoes)
    f1 = avaliador_f1.evaluate(predicoes)
    precision = avaliador_precision.evaluate(predicoes)
    recall = avaliador_recall.evaluate(predicoes)

    print(nome_modelo)
    print(f"Accuracy          : {accuracy:.4f}")
    print(f"F1-score          : {f1:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall   : {recall:.4f}")

    return {
        "modelo": nome_modelo,
        "accuracy": float(accuracy),
        "f1": float(f1),
        "weighted_precision": float(precision),
        "weighted_recall": float(recall)
    }

metricas_amanha = avaliar_classificacao(
    pred_amanha,
    coluna_label_amanha,
    "Rede Neural MLlib - Faixa térmica amanhã"
)

## 14. Treinamento da rede neural para faixa térmica dos próximos 7 dias

In [ ]:
num_features_semana = len(features_semana)
layers_semana = [num_features_semana, 32, 16, num_classes]

assembler_semana = VectorAssembler(
    inputCols=features_semana,
    outputCol="features_brutas",
    handleInvalid="keep"
)

scaler_semana = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withMean=False,
    withStd=True
)

mlp_semana = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=coluna_label_semana,
    predictionCol="prediction",
    layers=layers_semana,
    maxIter=100,
    blockSize=128,
    seed=42
)

pipeline_semana = Pipeline(stages=[assembler_semana, scaler_semana, mlp_semana])

modelo_mlp_semana = pipeline_semana.fit(semana_train_cls)

pred_semana = modelo_mlp_semana.transform(semana_test_cls)
pred_semana.createOrReplaceTempView("pred_semana")

print("Rede neural para faixa térmica média dos próximos 7 dias treinada com sucesso.")

## 15. Previsões do modelo dos próximos 7 dias

In [ ]:
pred_semana = spark.sql("""
    SELECT *,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria_real,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena'
            WHEN classe_temperatura_amanha = 2 THEN 'quente'
        END AS categoria_prevista,
        CASE
            WHEN classe_temperatura_semana = prediction THEN 1
            ELSE 0
        END AS acertou
    FROM pred_semana
""")

pred_semana.createOrReplaceTempView("pred_semana")

spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND(temperatura_media_proximos_7_dias, 2) AS temperatura_real,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_semana
    LIMIT 20
""").show(truncate=False)

metricas_semana = avaliar_classificacao(
    pred_semana,
    coluna_label_semana,
    "Rede Neural MLlib - Faixa térmica próximos 7 dias"
)

## 16. Comparação final das métricas

In [ ]:
metricas_mlp = spark.createDataFrame([metricas_amanha, metricas_semana])
metricas_mlp.createOrReplaceTempView("metricas_mlp")

spark.sql("""
    SELECT
        modelo,
        ROUND(accuracy, 4) AS accuracy,
        ROUND(f1, 4) AS f1,
        ROUND(weighted_precision, 4) AS weighted_precision,
        ROUND(weighted_recall, 4) AS weighted_recall
    FROM metricas_mlp
""").show(truncate=False)

### Gráfico: comparação da acurácia

In [ ]:
metricas_plot_df = spark.sql("""
    SELECT modelo, accuracy
    FROM metricas_mlp
    ORDER BY modelo
""")

fig = px.bar(
    spark_df_para_dicts(metricas_plot_df),
    x="modelo",
    y="accuracy",
    text="accuracy",
    title="Comparação da acurácia entre os modelos de rede neural",
    labels={"modelo": "Modelo", "accuracy": "Acurácia"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se o modelo dos próximos 7 dias apresentar acurácia maior, isso pode indicar que médias semanais geram classes mais estáveis.

## 17. Matriz de confusão

A matriz de confusão mostra onde o modelo acerta e onde ele erra.

Como as classes representam faixas de temperatura, é esperado que as confusões mais comuns ocorram entre categorias próximas.

In [ ]:
matriz_confusao_amanha = spark.sql("""
    SELECT categoria_real, categoria_prevista, COUNT(*) AS total
    FROM pred_amanha
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""")

matriz_confusao_semana = spark.sql("""
    SELECT categoria_real, categoria_prevista, COUNT(*) AS total
    FROM pred_semana
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""")

matriz_confusao_amanha.createOrReplaceTempView("matriz_confusao_amanha")
matriz_confusao_semana.createOrReplaceTempView("matriz_confusao_semana")

matriz_confusao_amanha.show(50, truncate=False)
matriz_confusao_semana.show(50, truncate=False)

### Gráfico: matriz de confusão — amanhã

In [ ]:
fig = px.density_heatmap(
    spark_df_para_dicts(matriz_confusao_amanha),
    x="categoria_prevista",
    y="categoria_real",
    z="total",
    text_auto=True,
    title="Matriz de confusão — faixa térmica amanhã",
    labels={
        "categoria_prevista": "Categoria prevista",
        "categoria_real": "Categoria real",
        "total": "Total"
    }
)

fig.show()

### Gráfico: matriz de confusão — próximos 7 dias

In [ ]:
fig = px.density_heatmap(
    spark_df_para_dicts(matriz_confusao_semana),
    x="categoria_prevista",
    y="categoria_real",
    z="total",
    text_auto=True,
    title="Matriz de confusão — faixa térmica próximos 7 dias",
    labels={
        "categoria_prevista": "Categoria prevista",
        "categoria_real": "Categoria real",
        "total": "Total"
    }
)

fig.show()

## 18. Recuperação das informações geográficas interpretáveis

Para criar gráficos e insights por região, vamos recuperar da base completa algumas colunas geográficas:

- `tipo_area`;
- `macro_regiao_sp`;
- `faixa_altitude`;
- `altitude`;
- `latitude`;
- `longitude`.

In [ ]:
contexto_amanha = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        altitude,
        latitude,
        longitude
    FROM dataset_amanha_completo
""")

contexto_semana = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        altitude,
        latitude,
        longitude
    FROM dataset_semana_completo
""")

contexto_amanha.createOrReplaceTempView("contexto_amanha")
contexto_semana.createOrReplaceTempView("contexto_semana")

pred_amanha_ctx = spark.sql("""
    SELECT p.*, c.tipo_area, c.macro_regiao_sp, c.faixa_altitude, c.altitude, c.latitude, c.longitude
    FROM pred_amanha p
    LEFT JOIN contexto_amanha c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_semana_ctx = spark.sql("""
    SELECT p.*, c.tipo_area, c.macro_regiao_sp, c.faixa_altitude, c.altitude, c.latitude, c.longitude
    FROM pred_semana p
    LEFT JOIN contexto_semana c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")
pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

In [ ]:
spark.sql("""
    SELECT
        station,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_amanha_ctx
    LIMIT 20
""").show(truncate=False)

## 19. Acurácia por tipo de área

Esta análise permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

In [ ]:
acuracia_tipo_area_amanha = spark.sql("""
    SELECT
        tipo_area,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    GROUP BY tipo_area
    ORDER BY tipo_area
""")

acuracia_tipo_area_semana = spark.sql("""
    SELECT
        tipo_area,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_semana_ctx
    GROUP BY tipo_area
    ORDER BY tipo_area
""")

acuracia_tipo_area_amanha.createOrReplaceTempView("acuracia_tipo_area_amanha")
acuracia_tipo_area_semana.createOrReplaceTempView("acuracia_tipo_area_semana")

acuracia_tipo_area_amanha.show(truncate=False)
acuracia_tipo_area_semana.show(truncate=False)

### Gráfico: acurácia por tipo de área — amanhã

In [ ]:
fig = px.bar(
    spark_df_para_dicts(acuracia_tipo_area_amanha),
    x="tipo_area",
    y="accuracy",
    text="accuracy",
    title="Acurácia por tipo de área — faixa térmica amanhã",
    labels={"tipo_area": "Tipo de área", "accuracy": "Acurácia"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Gráfico: acurácia por tipo de área — próximos 7 dias

In [ ]:
fig = px.bar(
    spark_df_para_dicts(acuracia_tipo_area_semana),
    x="tipo_area",
    y="accuracy",
    text="accuracy",
    title="Acurácia por tipo de área — faixa térmica próximos 7 dias",
    labels={"tipo_area": "Tipo de área", "accuracy": "Acurácia"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se a acurácia for menor em `serra_altitude`, isso pode indicar que a dinâmica térmica dessas áreas é mais específica por causa do relevo e da altitude.

Se a acurácia for menor em `litoral`, pode haver influência de umidade, brisa marítima e menor amplitude térmica.

Se a acurácia for menor em `urbano_metropolitano`, pode ser um indício de que efeitos locais, como concentração urbana e ilha de calor, são relevantes para a classificação térmica.

## 20. Distribuição das categorias reais por tipo de área

In [ ]:
classe_tipo_area_amanha = spark.sql("""
    SELECT
        tipo_area,
        categoria_real,
        COUNT(*) AS total_registros
    FROM pred_amanha_ctx
    GROUP BY tipo_area, categoria_real
    ORDER BY tipo_area, categoria_real
""")

classe_tipo_area_amanha.createOrReplaceTempView("classe_tipo_area_amanha")
classe_tipo_area_amanha.show(100, truncate=False)

In [ ]:
fig = px.bar(
    spark_df_para_dicts(classe_tipo_area_amanha),
    x="tipo_area",
    y="total_registros",
    color="categoria_real",
    barmode="group",
    title="Distribuição das categorias reais por tipo de área — amanhã",
    labels={
        "tipo_area": "Tipo de área",
        "total_registros": "Total de registros",
        "categoria_real": "Categoria real"
    }
)

fig.show()

### Insight esperado

A distribuição das categorias pode revelar padrões climáticos coerentes:

- áreas de `serra_altitude` tendem a concentrar mais registros mais_frias ou amenos;
- áreas `urbano_metropolitano` podem ter maior presença de categorias quentes;
- o `litoral` pode apresentar comportamento mais ameno ou estável;
- o `interior` pode apresentar maior amplitude térmica em determinadas épocas do ano.

## 21. Acurácia por mês e tipo de área

A análise mensal mostra se o modelo erra mais em determinados períodos do ano.

In [ ]:
pred_amanha_ctx = spark.sql("""
    SELECT *, MONTH(data_formatada) AS mes
    FROM pred_amanha_ctx
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

acuracia_mes_tipo_area_amanha = spark.sql("""
    SELECT
        mes,
        tipo_area,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    GROUP BY mes, tipo_area
    ORDER BY mes, tipo_area
""")

acuracia_mes_tipo_area_amanha.createOrReplaceTempView("acuracia_mes_tipo_area_amanha")
acuracia_mes_tipo_area_amanha.show(100, truncate=False)

In [ ]:
fig = px.line(
    spark_df_para_dicts(acuracia_mes_tipo_area_amanha),
    x="mes",
    y="accuracy",
    color="tipo_area",
    markers=True,
    title="Acurácia por mês e tipo de área — faixa térmica amanhã",
    labels={
        "mes": "Mês",
        "accuracy": "Acurácia",
        "tipo_area": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Se a acurácia cair em meses de transição, como outono e primavera, isso pode indicar que o modelo tem mais dificuldade em períodos de maior variabilidade atmosférica.

## 22. Acurácia por macro região

In [ ]:
acuracia_macro_amanha = spark.sql("""
    SELECT
        macro_regiao_sp,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    GROUP BY macro_regiao_sp
    ORDER BY macro_regiao_sp
""")

acuracia_macro_amanha.createOrReplaceTempView("acuracia_macro_amanha")
acuracia_macro_amanha.show(truncate=False)

In [ ]:
fig = px.bar(
    spark_df_para_dicts(acuracia_macro_amanha),
    x="macro_regiao_sp",
    y="accuracy",
    text="accuracy",
    title="Acurácia por macro região — faixa térmica amanhã",
    labels={"macro_regiao_sp": "Macro região", "accuracy": "Acurácia"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 23. Relação entre altitude e acerto

A altitude é uma variável importante para temperatura.

Este gráfico ajuda a observar se o modelo tem mais dificuldade em regiões mais altas, como áreas de serra.

Para manter o gráfico leve, a consulta usa uma amostra limitada.

In [ ]:
altitude_acerto = spark.sql("""
    SELECT
        altitude,
        tipo_area,
        acertou
    FROM pred_amanha_ctx
    WHERE altitude IS NOT NULL
      AND tipo_area IS NOT NULL
      AND acertou IS NOT NULL
    LIMIT 5000
""")

fig = px.scatter(
    spark_df_para_dicts(altitude_acerto),
    x="altitude",
    y="acertou",
    color="tipo_area",
    opacity=0.5,
    title="Relação entre altitude e acerto — faixa térmica amanhã",
    labels={
        "altitude": "Altitude (m)",
        "acertou": "Acertou a classe",
        "tipo_area": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Caso os acertos diminuam em altitudes maiores, isso pode indicar que regiões serranas possuem comportamento térmico mais específico.

Mesmo com a variável `altitude` presente no modelo, fatores locais como relevo, cobertura vegetal e massas de ar podem influenciar a temperatura de maneira mais complexa.

## 24. Salvamento das predições, métricas e matrizes de confusão

As saídas do notebook são salvas para comparação posterior com outros modelos.

Como este notebook trata redes neurais como classificação, as métricas salvas são diferentes das métricas de regressão.

In [ ]:
predicoes_mlp_amanha_saida = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        temperatura_amanha,
        classe_temperatura_amanha,
        categoria_real,
        prediction,
        categoria_prevista,
        acertou
    FROM pred_amanha_ctx
""")

predicoes_mlp_semana_saida = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area,
        macro_regiao_sp,
        faixa_altitude,
        temperatura_media_proximos_7_dias,
        classe_temperatura_semana,
        categoria_real,
        prediction,
        categoria_prevista,
        acertou
    FROM pred_semana_ctx
""")

predicoes_mlp_amanha_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_mlp_amanha"
)

predicoes_mlp_semana_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_mlp_semana"
)

metricas_mlp.write.mode("overwrite").parquet(
    f"{resultados_path}/metricas_redes_neurais_mlp"
)

matriz_confusao_amanha.write.mode("overwrite").parquet(
    f"{resultados_path}/matriz_confusao_mlp_amanha"
)

matriz_confusao_semana.write.mode("overwrite").parquet(
    f"{resultados_path}/matriz_confusao_mlp_semana"
)

print("Predições, métricas e matrizes de confusão salvas com sucesso.")

## 25. Salvamento dos modelos treinados

In [ ]:
modelo_mlp_amanha.write().overwrite().save(
    f"{modelos_path}/mlp_faixa_termica_amanha"
)

modelo_mlp_semana.write().overwrite().save(
    f"{modelos_path}/mlp_faixa_termica_semana"
)

print("Modelos de redes neurais salvos com sucesso.")

## 26. Conclusão

Neste notebook foram treinados dois modelos de **Redes Neurais com PySpark MLlib**:

- um modelo para classificar a faixa térmica de amanhã;
- um modelo para classificar a faixa térmica média dos próximos 7 dias.

Como o MLlib possui rede neural nativa para classificação, o alvo contínuo de temperatura foi convertido em categorias:

- `frio`;
- `ameno`;
- `quente`;
- `muito_quente`.

As bases utilizadas vieram diretamente do pré-processamento, já com:

- split temporal;
- imputação de valores ausentes;
- indexação de variáveis categóricas;
- features temporais de defasagem e janelas móveis.

A avaliação foi feita com:

- Accuracy;
- F1-score;
- Weighted Precision;
- Weighted Recall;
- Matriz de confusão.

Além disso, foram gerados gráficos com Plotly para analisar:

- distribuição das classes;
- comparação geral das métricas;
- matriz de confusão;
- acurácia por tipo de área;
- distribuição das categorias reais por tipo de área;
- acurácia por mês;
- acurácia por macro região;
- relação entre altitude e acerto.

A análise por `tipo_area` permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

A rede neural não entende sequência temporal automaticamente. Por isso, as features de defasagem e janelas móveis criadas no pré-processamento foram fundamentais para fornecer contexto temporal ao modelo.

Este notebook complementa os modelos de regressão, pois em vez de prever a temperatura exata em °C, classifica a condição térmica futura em faixas interpretáveis.